# 00 · Preparação do ambiente

Rode este notebook **antes da aula** (leva menos de 1 minuto). Ele instala as dependências e gera os dados sintéticos usados no curso, em `dados/`.

Todos os dados são gerados localmente pelo próprio DuckDB, então o curso funciona **sem internet** depois deste passo.

| Arquivo | Conteúdo | Linhas |
|---|---|---|
| `dados/clientes.csv` | cadastro de clientes | 50 mil |
| `dados/produtos.json` | catálogo de produtos | 2 mil |
| `dados/vendas.parquet` | itens vendidos (2024 a 2025), 3 itens por pedido | 5 milhões (ajustável) |
| `dados/vendas_2025_01.csv` | um mês de vendas em CSV | ~200 mil |
| `dados/precos_historico.parquet` | histórico de preços (para ASOF JOIN) | ~20 mil |
| `dados/logs/*.jsonl` | logs de uma API web | 600 mil |
| `dados/sensores/*.parquet` | leituras de 12 prensas industriais a cada 10 s | 311 mil |
| `dados/lake/vendas/ano=…/mes=…/` | vendas particionadas (Hive) | 5 milhões |

In [ ]:
# Instale uma vez (descomente se precisar)
# %pip install -q "duckdb>=2.0" duckdb-cli pandas polars pyarrow matplotlib

In [1]:
import duckdb, pandas, polars, pyarrow, os, time
print("duckdb ", duckdb.__version__)
print("pandas ", pandas.__version__)
print("polars ", polars.__version__)
print("pyarrow", pyarrow.__version__)

duckdb  2.0.0
pandas  3.1.1
polars  2.1.0
pyarrow 26.0.1


## Tamanho dos dados

`N_VENDAS` controla o tamanho da tabela de vendas. 5 milhões gera ~100 MB em Parquet e roda bem em qualquer notebook com 8 GB de RAM. Para máquinas fracas use 1 milhão.

In [2]:
N_VENDAS = 5_000_000
os.makedirs("dados/logs", exist_ok=True)
os.makedirs("dados/sensores", exist_ok=True)
os.makedirs("dados/lake", exist_ok=True)
con = duckdb.connect()
_ = con.execute("SELECT setseed(0.42)").fetchall()

### Clientes (CSV)

In [3]:
t0 = time.time()
con.sql("""
CREATE OR REPLACE TABLE clientes AS
SELECT
    i AS cliente_id,
    ['Ana','Bruno','Carla','Diego','Elisa','Felipe','Gabriela','Hugo','Isabela','João',
     'Karina','Lucas','Mariana','Nicolas','Olívia','Pedro','Rafaela','Samuel','Tatiane','Vitor']
        [1 + floor(random()*20)::INT] || ' ' ||
    ['Silva','Souza','Oliveira','Santos','Lima','Pereira','Costa','Almeida','Ferreira','Rodrigues']
        [1 + floor(random()*10)::INT]                                   AS nome,
    c.cidade, c.uf,
    DATE '2020-01-01' + (random()*1800)::INT                      AS data_cadastro,
    CASE WHEN random() < 0.15 THEN 'empresa' ELSE 'pessoa_fisica' END AS segmento
FROM (
    SELECT i, [
        {'cidade':'Recife','uf':'PE'},{'cidade':'São Paulo','uf':'SP'},{'cidade':'Rio de Janeiro','uf':'RJ'},
        {'cidade':'Belo Horizonte','uf':'MG'},{'cidade':'Salvador','uf':'BA'},{'cidade':'Fortaleza','uf':'CE'},
        {'cidade':'Curitiba','uf':'PR'},{'cidade':'Porto Alegre','uf':'RS'},{'cidade':'Manaus','uf':'AM'},
        {'cidade':'Brasília','uf':'DF'},{'cidade':'Goiânia','uf':'GO'},{'cidade':'Belém','uf':'PA'},
        {'cidade':'João Pessoa','uf':'PB'},{'cidade':'Natal','uf':'RN'},{'cidade':'Florianópolis','uf':'SC'},
        {'cidade':'Campinas','uf':'SP'}
    ][1 + floor(random()*16)::INT] AS c
    FROM range(1, 50001) t(i)
)
""")
con.sql("COPY clientes TO 'dados/clientes.csv' (HEADER)")
print(f"clientes ok em {time.time()-t0:.1f}s")

clientes ok em 0.0s


### Produtos (JSON)

In [4]:
con.sql("""
CREATE OR REPLACE TABLE produtos AS
SELECT
    i AS produto_id,
    cat.categoria || ' ' || cat.prefixo || '-' || lpad(i::VARCHAR, 4, '0') AS nome,
    cat.categoria,
    round(cat.preco_base * (0.5 + random()*1.5), 2)                     AS preco,
    {'peso_kg': round(random()*5, 2), 'cor': ['preto','branco','azul','vermelho'][1 + floor(random()*4)::INT]} AS atributos
FROM (
    SELECT i, [
        {'categoria':'Filtros de ar','prefixo':'FA','preco_base':89.0},
        {'categoria':'Eletrônicos','prefixo':'EL','preco_base':899.0},
        {'categoria':'Casa','prefixo':'CA','preco_base':149.0},
        {'categoria':'Esporte','prefixo':'ES','preco_base':199.0},
        {'categoria':'Livros','prefixo':'LV','preco_base':59.0},
        {'categoria':'Beleza','prefixo':'BE','preco_base':79.0},
        {'categoria':'Ferramentas','prefixo':'FE','preco_base':249.0},
        {'categoria':'Brinquedos','prefixo':'BR','preco_base':119.0}
    ][1 + floor(random()*8)::INT] AS cat
    FROM range(1, 2001) t(i)
)
""")
con.sql("COPY produtos TO 'dados/produtos.json' (FORMAT json, ARRAY true)")
con.sql("FROM produtos LIMIT 3").show()

┌────────────┬─────────────────────┬─────────────┬────────┬─────────────────────────────────────┐
│ produto_id │        nome         │  categoria  │ preco  │              atributos              │
│   int64    │       varchar       │   varchar   │ double │ struct(peso_kg double, cor varchar) │
├────────────┼─────────────────────┼─────────────┼────────┼─────────────────────────────────────┤
│          1 │ Esporte ES-0001     │ Esporte     │ 271.81 │ {'peso_kg': 0.92, 'cor': vermelho}  │
│          2 │ Esporte ES-0002     │ Esporte     │ 318.88 │ {'peso_kg': 2.26, 'cor': branco}    │
│          3 │ Ferramentas FE-0003 │ Ferramentas │ 476.14 │ {'peso_kg': 0.7, 'cor': azul}       │
└────────────┴─────────────────────┴─────────────┴────────┴─────────────────────────────────────┘



### Vendas (Parquet, CSV e lake particionado)

In [5]:
t0 = time.time()
con.sql(f"""
CREATE OR REPLACE TABLE vendas AS
WITH ciclo AS (
    -- cada cliente começa a comprar numa data e "vive" um tempo (churn exponencial)
    SELECT cliente_id,
           TIMESTAMP '2024-01-01' + to_seconds((random() * 700 * 86400)::BIGINT) AS inicio,
           -ln(random() + 1e-9) * 300                                         AS vida_dias
    FROM clientes
),
pedidos AS (
    SELECT
        p                                                   AS pedido_id,
        -- poucos clientes compram muito (distribuição enviesada)
        1 + floor(pow(random(), 2.2) * 50000)::INT          AS cliente_id,
        random()                                            AS u,
        ['site','app','marketplace','loja_fisica'][1 + floor(random()*4)::INT] AS canal,
        CASE WHEN random() < 0.04 THEN 'cancelado'
             WHEN random() < 0.03 THEN 'devolvido'
             ELSE 'concluido' END                           AS status
    FROM range(1, {N_VENDAS} // 3 + 1) t(p)
),
pedidos_datados AS (
    SELECT p.* EXCLUDE (u),
           c.inicio + to_seconds((p.u * least(c.vida_dias * 86400,
                 epoch(TIMESTAMP '2025-12-31 23:59:59' - c.inicio)))::BIGINT) AS data_venda
    FROM pedidos p JOIN ciclo c USING (cliente_id)
),
itens AS (
    SELECT pd.*,
        1 + floor(pow(random(), 1.6) * 2000)::INT           AS produto_id,
        1 + floor(pow(random(), 3) * 5)::INT                AS quantidade
    FROM pedidos_datados pd, range(3) r(k)
)
SELECT row_number() OVER (ORDER BY data_venda, pedido_id) AS venda_id,
       i.pedido_id, i.cliente_id, i.produto_id, i.data_venda, i.quantidade, i.canal, i.status,
       round(p.preco * (0.9 + random()*0.2), 2) AS preco_unitario
FROM itens i JOIN produtos p USING (produto_id)
ORDER BY data_venda
""")
con.sql("COPY vendas TO 'dados/vendas.parquet' (FORMAT parquet, COMPRESSION zstd)")
con.sql("""COPY (FROM vendas WHERE data_venda >= '2025-01-01' AND data_venda < '2025-02-01')
           TO 'dados/vendas_2025_01.csv' (HEADER, DELIMITER ';')""")
con.sql("""COPY (SELECT *, year(data_venda) AS ano, month(data_venda) AS mes FROM vendas)
           TO 'dados/lake/vendas' (FORMAT parquet, PARTITION_BY (ano, mes), OVERWRITE_OR_IGNORE)""")
print(f"vendas ok em {time.time()-t0:.1f}s")
con.sql("SELECT count(*) AS linhas, min(data_venda), max(data_venda) FROM vendas").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

vendas ok em 5.9s
┌─────────┬─────────────────────┬─────────────────────┐
│ linhas  │   min(data_venda)   │   max(data_venda)   │
│  int64  │      timestamp      │      timestamp      │
├─────────┼─────────────────────┼─────────────────────┤
│ 4999998 │ 2024-01-01 08:39:15 │ 2025-12-31 23:58:46 │
└─────────┴─────────────────────┴─────────────────────┘



### Histórico de preços (para o ASOF JOIN)

In [6]:
con.sql("""
COPY (
    SELECT produto_id,
           TIMESTAMP '2024-01-01' + INTERVAL (k * 73) DAY  AS valido_desde,
           round(preco * (1 + (k * 0.02) + (random() - 0.5) * 0.1), 2) AS preco_tabela
    FROM produtos, range(10) r(k)
) TO 'dados/precos_historico.parquet'
""")

CountBIGINT
20000


### Logs de API (JSON Lines, um arquivo por dia)

In [7]:
con.sql("""
CREATE OR REPLACE TABLE logs AS
SELECT
    TIMESTAMP '2025-03-01' + to_seconds((random() * 3 * 86400)::BIGINT) AS ts,
    ['GET /produtos','GET /produtos/{id}','POST /carrinho','POST /checkout','GET /busca','POST /login']
        [1 + floor(pow(random(), 1.4) * 6)::INT]                          AS rota,
    ['api-1','api-2','api-3'][1 + floor(random()*3)::INT]                 AS servidor,
    1 + floor(random() * 3000)::INT                                      AS usuario_id,
    pow(random(), 4)                                                     AS r_lat
FROM range(600000)
""")
con.sql("""
CREATE OR REPLACE TABLE logs AS
SELECT * EXCLUDE (r_lat),
    round(CASE rota WHEN 'GET /busca' THEN 80 WHEN 'POST /checkout' THEN 150 WHEN 'POST /login' THEN 60 ELSE 25 END
          * (1 + 20 * r_lat)
          * CASE WHEN rota = 'POST /checkout' AND servidor = 'api-3' THEN 2.5 ELSE 1 END, 1) AS latencia_ms,
    CASE WHEN rota = 'POST /checkout' AND servidor = 'api-3' AND random() < 0.12 THEN 500
         WHEN random() < 0.01 THEN 500
         WHEN random() < 0.05 THEN 404
         ELSE 200 END AS status,
    CASE WHEN random() < 0.3 THEN {'versao_app': '3.' || floor(random()*4)::INT, 'mobile': true}
         ELSE {'versao_app': NULL, 'mobile': false} END AS contexto
FROM logs ORDER BY ts
""")
for dia in ['2025-03-01', '2025-03-02', '2025-03-03']:
    con.sql(f"""COPY (FROM logs WHERE ts::DATE = '{dia}')
               TO 'dados/logs/app_{dia}.jsonl' (FORMAT json)""")
print(os.listdir("dados/logs"))

['app_2025-03-03.jsonl', 'app_2025-03-01.jsonl', 'app_2025-03-02.jsonl']


### Sensores industriais (Parquet, um arquivo por dia)
Doze prensas em três linhas de produção, com vibração, temperatura e pressão a cada 10 segundos.

In [8]:
con.sql("""
CREATE OR REPLACE TABLE sensores AS
WITH base AS (
    SELECT
        TIMESTAMP '2025-03-01' + to_seconds(s) AS ts,
        s,
        m AS id,
        'prensa-' || lpad(m::VARCHAR, 2, '0') AS maquina,
        ['A', 'B', 'C'][1 + ((m - 1) // 4)::INT] AS linha,
        -- desgaste do rolamento da prensa-07: começa no 2º dia às 06h e cresce até o fim
        CASE WHEN m = 7 THEN greatest(0, (s - 108000) / (259200 - 108000)) ELSE 0 END AS desgaste,
        sin(2 * pi() * s / 86400) AS ciclo
    FROM range(0, 259200, 10) t(s), range(1, 13) mm(m)
)
SELECT ts, linha, maquina,
    round(2.2 + 0.1 * id / 12 + 0.5 * ciclo + (random() - 0.5) * 1.2
          + desgaste * (6 + 3 * random()), 2)                          AS vibracao_mm_s,
    round(58 + 4 * ciclo + (random() - 0.5) * 3 + desgaste * 18, 1)      AS temperatura_c,
    round(120 + (random() - 0.5) * 8 - desgaste * 6, 1)                AS pressao_bar
FROM base
ORDER BY ts, maquina
""")
for dia in ['2025-03-01', '2025-03-02', '2025-03-03']:
    con.sql(f"""COPY (FROM sensores WHERE ts::DATE = '{dia}')
               TO 'dados/sensores/leituras_{dia}.parquet' (FORMAT parquet)""")
print(sorted(os.listdir("dados/sensores")))

['leituras_2025-03-01.parquet', 'leituras_2025-03-02.parquet', 'leituras_2025-03-03.parquet']


In [9]:
for raiz, _, arquivos in os.walk("dados"):
    if "lake" in raiz:
        continue
    for a in sorted(arquivos):
        p = os.path.join(raiz, a)
        print(f"{p:45s} {os.path.getsize(p)/1e6:8.1f} MB")
con.close()

dados/clientes.csv                                 2.9 MB
dados/precos_historico.parquet                     0.1 MB
dados/produtos.json                                0.3 MB
dados/vendas.parquet                              51.9 MB
dados/vendas_2025_01.csv                          14.5 MB
dados/logs/app_2025-03-01.jsonl                   33.5 MB
dados/logs/app_2025-03-02.jsonl                   33.6 MB
dados/logs/app_2025-03-03.jsonl                   33.5 MB
dados/sensores/leituras_2025-03-01.parquet         0.4 MB
dados/sensores/leituras_2025-03-02.parquet         0.4 MB
dados/sensores/leituras_2025-03-03.parquet         0.4 MB
